<a href="https://colab.research.google.com/github/pesquisaitr-cmd/PNAD/blob/main/PNAD_ETL_NACIONAL_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook ETL Nacional PNAD/CAGED para Dashboard Interativo

Este notebook Python foi projetado para ser executado no Google Colab, aproveitando seus recursos computacionais e a integração facilitada com o Google Drive.

**Funcionalidades:**
- Montagem do Google Drive para acesso aos dados.
- Extração e processamento dos índices INPC para deflacionamento de salários.
- Consolidação de dados de todos os municípios do Brasil.
- Agrupamento por Região, UF, Município, CBO, Mês/Ano, Tipo de Movimentação, Sexo e Raça/Cor.
- Geração de um arquivo `.parquet` otimizado para o dashboard.

**Pré-requisitos:**
1.  **Arquivos CAGED**: Coloque seus arquivos de microdados do CAGED (.txt) na pasta `My Drive/PNAD_Dashboard/dados/caged/` no seu Google Drive.
2.  **Arquivo INPC**: Opcionalmente, coloque um arquivo `indices.csv` com os dados históricos do INPC na mesma pasta. Se não for fornecido, o notebook tentará fazer o scraping do site `dadosdemercado.com.br`.

In [ ]:
# 1. Instalar bibliotecas necessárias
# (Execute esta célula apenas uma vez ou se as bibliotecas não estiverem instaladas)

In [37]:
!pip install pandas numpy requests beautifulsoup4 py7zr fastparquet
!pip install openpyxl

In [ ]:
# 2. Importar bibliotecas

In [38]:
import pandas as pd
import numpy as np
import os
import glob
import requests
from bs4 import BeautifulSoup
from google.colab import drive

In [ ]:
# 3. Montar Google Drive
# Isso permitirá que o Colab acesse seus arquivos no Google Drive.
# Uma janela de autenticação será aberta. Siga as instruções.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 4. Configurações de Caminhos
# Certifique-se de que a estrutura de pastas no seu Google Drive seja a seguinte:
# My Drive/PNAD_Dashboard/dados/caged/ (para os arquivos .txt do CAGED e indices.csv)
# My Drive/PNAD_Dashboard/dados/ (para o arquivo .parquet de saída)

In [52]:
CAMINHO_BASE_DRIVE = "/content/drive/My Drive/PNAD_Dashboard"
CAMINHO_DADOS = os.path.join(CAMINHO_BASE_DRIVE, "dados")
CAMINHO_CAGED = os.path.join(CAMINHO_DADOS, "caged")
CAMINHO_PROCESSADO = CAMINHO_DADOS

print(CAMINHO_DADOS)
print(CAMINHO_CAGED)
print(CAMINHO_PROCESSADO)


/content/drive/My Drive/PNAD_Dashboard/dados
/content/drive/My Drive/PNAD_Dashboard/dados/caged
/content/drive/My Drive/PNAD_Dashboard/dados


In [ ]:
# 5. Função para Extrair Índices INPC

In [20]:
def extrair_indices_inpc():
    caminho_inpc = os.path.join(CAMINHO_CAGED, "indices.csv")
    df_inpc = pd.DataFrame()
    print(f"Lendo INPC de {caminho_inpc}...")
    df_inpc = pd.read_csv(caminho_inpc)

    inpc_long = df_inpc.melt(
        id_vars=["Ano"],
        value_vars=["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"],
        var_name="mes_nome",
        value_name="inpc_val"
    )

    mapa_meses = {
        "Jan":"01","Fev":"02","Mar":"03","Abr":"04","Mai":"05","Jun":"06",
        "Jul":"07","Ago":"08","Set":"09","Out":"10","Nov":"11","Dez":"12"
    }
    inpc_long["mes"] = inpc_long["mes_nome"].map(mapa_meses)
    inpc_long["Anomes"] = inpc_long["Ano"].astype(str) + inpc_long["mes"]
    inpc_long["inpc_val"] = inpc_long["inpc_val"].astype(str).str.replace(",", ".").str.replace("%", "")
    inpc_long["inpc_val"] = pd.to_numeric(inpc_long["inpc_val"], errors="coerce")

    return inpc_long[["Anomes", "inpc_val"]].dropna()


In [22]:
df_inpc = extrair_indices_inpc()

print(df_inpc.head(30))   # mostra as primeiras linhas
print(df_inpc.shape)    # mostra número de linhas e colunas

Lendo INPC de /content/drive/My Drive/PNAD_Dashboard/dados/caged/indices.csv...
    Anomes  inpc_val
0   202601      0.39
1   202501      0.00
2   202401      0.57
3   202301      0.46
4   202201      0.67
5   202101      0.27
6   202001      0.19
7   201901      0.36
8   201801      0.23
9   201701      0.42
10  201601      1.51
11  201501      1.48
12  201401      0.63
13  201301      0.92
14  201201      0.51
15  201101      0.94
16  201001      0.88
17  200901      0.64
18  200801      0.69
19  200701      0.49
20  200601      0.38
21  200501      0.57
22  200401      0.83
23  200301      2.47
24  200201      1.07
25  200101      0.77
26  200001      0.61
27  202602      0.56
28  202502      1.48
29  202402      0.81
(316, 2)


In [ ]:
# 6. Função Principal de ETL Nacional

In [55]:
arquivo_parquet = os.path.join(CAMINHO_DADOS, "indices.parquet")

# Leitura do parquet
df = pd.read_parquet(arquivo_parquet)

# Conferindo os dados
print(df.shape)   # mostra número de linhas e colunas
print(df.head())  # mostra as primeiras linhas
print(df.info())  # mostra tipos de dados

(27, 1)
  Ano,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez,Ano
0  2026,"0,39%","0,56%","0,91%","0,81%",,,,,,,,,"...     
1  2025,"0,0%","1,48%","0,51%","0,48%","0,35%","0...     
2  2024,"0,57%","0,81%","0,19%","0,37%","0,46%","...     
3  2023,"0,46%","0,77%","0,64%","0,53%","0,36%","...     
4  2022,"0,67%","1,0%","1,71%","1,04%","0,45%","0...     
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 1 columns):
 #   Column                                                   Non-Null Count  Dtype 
---  ------                                                   --------------  ----- 
 0   Ano,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez,Ano  27 non-null     object
dtypes: object(1)
memory usage: 348.0+ bytes
None


In [54]:
def processar_etl_nacional():
    COLUNAS_NECESSARIAS = [
        "competênciamov", "região", "uf", "município", "cbo2002ocupação",
        "sexo", "raçacor", "saldomovimentação", "salário" ]

    mapa_regiao = {1: "Norte", 2: "Nordeste", 3: "Sudeste", 4: "Sul", 5: "Centro-Oeste"}
    mapa_uf = {
        11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO",
        21: "MA", 22: "PI", 23: "CE", 24: "RN", 25: "PB", 26: "PE", 27: "AL", 28: "SE", 29: "BA",
        31: "MG", 32: "ES", 33: "RJ", 35: "SP",
        41: "PR", 42: "SC", 43: "RS",
        50: "MS", 51: "MT", 52: "GO", 53: "DF"}

    mapa_municipio = {}

    # Associando nume do município ao seu código - arquivo códigos_municípios.xlsx
    try:
        arquivos_xlsx = glob.glob(os.path.join(CAMINHO_CAGED, "*.xlsx"))
        if arquivos_xlsx:
           arquivo_excel = arquivos_xlsx[0]  # pega o primeiro arquivo encontrado
           df_mun = pd.read_excel(arquivo_excel, engine="openpyxl")
           mapa_municipio = dict(zip(df_mun["Código_IBGE"], df_mun["Município"]))
           print("Mapa de municípios carregado com sucesso!")
        else:
            raise FileNotFoundError("Nenhum arquivo .xlsx encontrado na pasta.")
    except Exception as e:
        print("Erro ao ler Excel:", e)
        raise
    ##############################################################################

    mapa_contrato = {-1: "Demitidos", 1: "Admitidos"}
    mapa_sexo = {1: "Masculino", 3: "Feminino", 9: "Não Identificado"}
    mapa_raca = {1: "Branca", 2: "Preta", 3: "Parda", 4: "Amarela", 5: "Indígena",
                 6: "Não Informado", 9: "Não Identificado"}

    # lEITURA do arquivo com os 12 meses de informações
    arquivo_parquet = os.path.join(CAMINHO_DADOS, "df_final.parquet")
    # Leitura do parquet
    df_final = pd.read_parquet(arquivo_parquet)

    df_inpc = extrair_indices_inpc()
    df_inpc["inpc_fator"] = 1 + df_inpc["inpc_val"] / 100
    df_inpc = df_inpc.sort_values("Anomes")
    df_inpc["inpc_acm"] = df_inpc["inpc_fator"].cumprod()

    df_agrupado = df_final.groupby(["região", "uf", "município", "cbo2002ocupação",
                               "competênciamov", "saldomovimentação", "sexo", "raçacor"],
                                as_index=False).agg(Media_Salario=("salário", "mean"),
                                n_movimentacoes=("salário", "size"))

    df_final = pd.concat(lista_consolidada, ignore_index=True)
    df_final["contrato"] = df_final["saldomovimentação"].map(mapa_contrato)
    df_final["genero"] = df_final["sexo"].map(mapa_sexo)
    df_final["etnia"] = df_final["raçacor"].map(mapa_raca)
    df_final["nome_regiao"] = df_final["região"].map(mapa_regiao)
    df_final["sigla_uf"] = df_final["uf"].map(mapa_uf)
    df_final["Município"] = df_final["município"].map(mapa_municipio).fillna(df_final["município"]).astype(str)
    df_final["Anomes"] = df_final["competênciamov"].astype(str)
    df_final = pd.merge(df_final, df_inpc[["Anomes", "inpc_acm"]], on="Anomes", how="left")
    df_final["Sal_def_INPC"] = df_final["Media_Salario"] / df_final["inpc_acm"]

    df_dashboard = df_final[["Anomes", "nome_regiao", "sigla_uf", "nome_municipio",
                  "cbo2002ocupação", "contrato", "genero", "etnia", "Media_Salario",
                  "Sal_def_INPC", "n_movimentacoes"]].rename(columns={"cbo2002ocupação": "cbo"})
    df_dashboard.to_parquet(os.path.join(CAMINHO_PROCESSADO, "dados_dashboard_nacional.parquet"), index=False)
    print("ETL CONCLUÍDO!")

if __name__ == "__main__":
    processar_etl_nacional()

Mapa de municípios carregado com sucesso!
Lendo INPC de /content/drive/My Drive/PNAD_Dashboard/dados/caged/indices.csv...
Processando competênciamov...


FileNotFoundError: [Errno 2] No such file or directory: 'competênciamov'